# Three-target second-step sufficiency development test

Based on the supplied working notebook. V2 questions, previous selector, original asset descriptions, dependencies, generation and answer routing are preserved. Only the iterative decision after one acquired observation receives a sufficiency/next-modality reminder. The first decision and one-shot AGENT are unchanged.

Default SECOND_STEP_REVIEW_ENABLED=True. False runs the unmodified-message control with a different label. Each run preserves the three-target/108-record workflow. Use a fresh runtime and an unused run label. If comparing runtimes, retain the same model revision, code revision and package versions. No additional model calls or acquisitions are added. Results are versioned development data; do not mix with formal runs.


## 1. Project and verified dataset


In [ ]:
import sys

assert sys.version_info >= (3, 12), sys.version

In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path("/content/autonomous-modality-selection")
if (PROJECT / ".git").is_dir():
    dirty = subprocess.check_output(
        ["git", "-C", str(PROJECT), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("The Colab checkout has uncommitted changes. Save them before updating.")
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "merge", "--ff-only", "origin/main"], check=True)
elif PROJECT.exists():
    raise RuntimeError(f"Directory exists but is not a Git checkout; inspect it first: {PROJECT}")
else:
    # The repository is public; no token is needed for cloning.
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/liaojiqichi/autonomous-modality-selection.git",
            str(PROJECT),
        ],
        check=True,
    )
print(
    "Current commit:",
    subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip(),
)

In [ ]:
%cd /content/autonomous-modality-selection
%pip install -e ".[dev,pilot,multispectral]"

In [ ]:
!python -m ruff check .
!python -m ruff format --check .
!python -m pytest

In [ ]:
from autonomous_modality.colab_four_modality import (
    download_archive,
    extract_dataset,
)
from autonomous_modality.multispectral_release import verify_release

DATA_MODE = "download"  # Alternatives: "upload" or "local" (e.g. a mounted Drive ZIP).
ARCHIVE_PATH = Path("/content/mercury-four-modality-20261006.zip")
if DATA_MODE == "download":
    download_archive(ARCHIVE_PATH)
elif DATA_MODE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one dataset ZIP.")
    filename = next(iter(uploaded))
    ARCHIVE_PATH = Path(filename).resolve()
    del uploaded  # Release the large uploaded byte buffer before loading the model.
elif DATA_MODE != "local":
    raise ValueError("Unknown DATA_MODE")
DATASET_ROOT = extract_dataset(ARCHIVE_PATH, Path("/content"))
release = verify_release(DATASET_ROOT)
print("Verified real development targets:", [(c.case_id, c.name) for c in release.cases])

In [ ]:
!nvidia-smi

## 2. Model environment
Select Qwen or Gemma before installation. Restart if loaded dependencies change.
Use a fresh runtime when switching models. GPU preflight remains required.


In [ ]:
import importlib.metadata
import subprocess
import sys

MODEL_ID = "google/gemma-4-E4B-it"  # @param ["Qwen/Qwen3-VL-8B-Instruct", "google/gemma-4-E4B-it"]
MODEL_CLASSES = {
    "Qwen/Qwen3-VL-8B-Instruct": "Qwen3VLForConditionalGeneration",
    "google/gemma-4-E4B-it": "Gemma4ForConditionalGeneration",
}
if MODEL_ID not in MODEL_CLASSES:
    raise ValueError(f"Unsupported model: {MODEL_ID}")
if "model" in globals():
    raise RuntimeError("Skip installation for a loaded model; switching requires a fresh runtime.")
packages = (
    ["transformers==4.57.6", "accelerate>=1.2,<2", "bitsandbytes>=0.48,<1", "Pillow==12.3.0"]
    if MODEL_ID == "Qwen/Qwen3-VL-8B-Instruct"
    else ["transformers==5.19.0", "accelerate==1.15.0", "bitsandbytes==0.50.2", "Pillow==12.3.0"]
)
if MODEL_ID == "google/gemma-4-E4B-it":
    packages += ["numpy==2.1.3", "rasterio==1.5.2", "matplotlib==3.10.0", "pydantic==2.13.5"]
loaded = {
    name: importlib.metadata.version(name)
    for name in (
        "transformers",
        "accelerate",
        "bitsandbytes",
        "Pillow",
        "numpy",
        "rasterio",
        "matplotlib",
        "pydantic",
    )
    if name in sys.modules or (name == "Pillow" and "PIL" in sys.modules)
}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
if any(importlib.metadata.version(name) != version for name, version in loaded.items()):
    raise RuntimeError("Libraries changed. Restart the runtime and rerun setup.")
print("Selected model:", MODEL_ID)
MODEL_REVISION = "main"  # Set a saved commit SHA to reproduce an earlier load.

In [ ]:
if "model" in globals():
    if str(globals()["model"].config._name_or_path) != MODEL_ID or "processor" not in globals():
        raise RuntimeError("Loaded model differs from the selection. Use a fresh runtime.")
    print(f"Reusing {MODEL_ID}.")
else:
    import bitsandbytes
    import torch
    import transformers
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
    )

    assert torch.cuda.is_available(), "Select a Colab GPU runtime first."

    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)
    print("Transformers:", transformers.__version__)
    print("bitsandbytes:", bitsandbytes.__version__)

    if MODEL_ID not in MODEL_CLASSES:
        raise ValueError(f"Unsupported model: {MODEL_ID}")
    if MODEL_ID == "Qwen/Qwen3-VL-8B-Instruct" and transformers.__version__ != "4.57.6":
        raise RuntimeError("The Qwen adapter requires Transformers 4.57.6.")
    model_class = getattr(transformers, MODEL_CLASSES[MODEL_ID], None)
    if model_class is None:
        raise RuntimeError("Selected model class unavailable. Update dependencies and restart.")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    from huggingface_hub import HfApi

    RESOLVED_MODEL_REVISION = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    if not RESOLVED_MODEL_REVISION:
        raise RuntimeError("Could not resolve a model revision.")
    backbone_attention = "sdpa"
    model = model_class.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation=backbone_attention,
        revision=RESOLVED_MODEL_REVISION,
        low_cpu_mem_usage=True,
    )
    model.eval()

    processor = AutoProcessor.from_pretrained(MODEL_ID, revision=RESOLVED_MODEL_REVISION)

    print(
        "GPU memory after model loading:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GiB",
    )

MODEL_NAME = str(model.config._name_or_path)
ATTENTION_POLICY = (
    "qwen-4.57.6-explicit-kv-efficient-sdpa"
    if MODEL_NAME == "Qwen/Qwen3-VL-8B-Instruct"
    else "gemma4-native-sdpa"
)
print("Attention policy:", ATTENTION_POLICY)
RESOLVED_MODEL_REVISION = globals().get(
    "RESOLVED_MODEL_REVISION", getattr(model.config, "_commit_hash", None)
)
if not RESOLVED_MODEL_REVISION:
    raise RuntimeError("Reused model revision is unknown; load in a fresh runtime.")
LOADED_ATTENTION_CONFIG = {
    name: getattr(getattr(model.config, name), "_attn_implementation", None)
    for name in ("text_config", "vision_config", "audio_config")
    if getattr(model.config, name, None) is not None
}
print("Model and processor revision:", RESOLVED_MODEL_REVISION)
print("Loaded attention:", LOADED_ATTENTION_CONFIG)

In [ ]:
image_processor = processor.image_processor
if MODEL_ID == "Qwen/Qwen3-VL-8B-Instruct":
    if not isinstance(getattr(image_processor, "size", None), dict):
        raise RuntimeError("Unexpected Qwen image processor configuration.")
    image_processor.size = {
        **image_processor.size,
        "shortest_edge": 128 * 128,
        "longest_edge": 320 * 320,
    }
    if hasattr(image_processor, "min_pixels"):
        image_processor.min_pixels = 128 * 128
    if hasattr(image_processor, "max_pixels"):
        image_processor.max_pixels = 320 * 320
else:
    if not hasattr(image_processor, "max_soft_tokens"):
        raise RuntimeError("Unexpected Gemma image processor configuration.")
    image_processor.max_soft_tokens = 280
print(image_processor.to_json_string())

## Development settings and versioned questions

In [ ]:
from pathlib import Path

TARGET_IDS = [16604, 5031, 2179]  # Eminescu, Nampeyo, Thoreau; development only.
SELECTOR_VARIANT = "previous"  # Fixed for this ablation.
SECOND_STEP_REVIEW_ENABLED = True  # @param {type:"boolean"}
SECOND_STEP_REVIEW_PROMPT = (
    "Review the acquired evidence against the question and answer requirements. Is it "
    "sufficient for concrete analytical approaches and feature-grounded explanations? "
    "Identify an unresolved issue using a specific observation or representation "
    "limitation. Compare stopping with each remaining feasible modality: would it "
    "address that issue, enable a distinct analysis, or help distinguish explanations, "
    "rather than repeat support? Request the clearest expected contribution, or FINISH "
    "if further acquisition is unlikely to help within budget. Judge unaccessed "
    "evidence only by declared capabilities. Previously accessed data remain charged; "
    "no replacement or refund. Keep the existing action JSON schema and reason-length "
    "limit; briefly justify the action."
)
RUN_LABEL = (
    "three-target-v2-second-step-unified-review-001"
    if SECOND_STEP_REVIEW_ENABLED
    else "three-target-v2-second-step-unified-control-001"
)
RESULTS_ROOT = Path("/content/experiment-results")  # A mounted Drive path is also supported.

if type(SECOND_STEP_REVIEW_ENABLED) is not bool:
    raise ValueError("SECOND_STEP_REVIEW_ENABLED must be boolean.")
if SELECTOR_VARIANT != "previous":
    raise ValueError("Unknown selector variant.")
if TARGET_IDS != [16604, 5031, 2179]:
    raise ValueError("This notebook is restricted to the three development targets.")
if not RUN_LABEL or Path(RUN_LABEL).name != RUN_LABEL:
    raise ValueError("RUN_LABEL must be one directory name.")
print("Expected: 3 targets x 4 questions x 9 attempts = 108 records per model.")

# Shared settings applied to every target and condition.
BUDGET = 4  # Primary budget 4; use 3 for sensitivity analysis.
SEED = 42
MAX_MODALITIES = 2
RANDOM_DRAWS = 5
IMAGE_EDGE = 448
ANSWER_TOKENS = 1024
SELECTOR_TOKENS = 512
MAX_INPUT_TOKENS = 8192
RECORD_VERSION = "colab-ap-four-modality-en-1"
ANSWER_REPETITION_PENALTY = 1.2
ITERATIVE_SELECTOR_TOKENS = 512
IS_DEVELOPMENT = True  # Eminescu has already been inspected and is development-only.

In [ ]:
DEVELOPMENT_SETTINGS = {
    "alignment": "Choose inputs for the specific question and its answer requirements. A "
    "means distinct analytical approaches feasible with supplied "
    "representations. P means mechanisms or alternative explanations linked "
    "to supplied features. Compare feasible combinations by their additional "
    "contribution to A and P, including information lost when another "
    "modality is omitted. Optical views support qualitative structure, "
    "texture, boundaries and spatial relations. Terrain supports relief and "
    "profile comparisons; multispectral summaries support regional spectral "
    "comparisons; catalogue records support historical classification "
    "checks. Use the inventory to judge actual availability and limitations. "
    "Numerical inputs are not inherently more useful than visual evidence. "
    "Optical brightness does not measure elevation or composition. No "
    "modality is mandatory; spending more receives no separate reward.",
    "complementarity": "After observing the first package, identify what remains "
    "unresolved. Request a second modality when it offers a distinct "
    "analytical route or helps distinguish explanations; otherwise "
    "finish. Retain the original action schema and cumulative access "
    "limits.",
    "questions": {
        "version": "mercury-selection-value-dev-2.0-en",
        "split": "development",
        "questions": [
            {
                "question_id": "Q1",
                "question_type": "ANALYTICAL_APPROACH_DISCOVERY",
                "text": "How well does the historical morphology "
                "classification of {crater} describe its present "
                "visible structure? Identify a classification "
                "claim worth checking, propose ways to assess "
                "it, and explain what could account for "
                "agreement or disagreement.",
                "answer_requirements": [
                    "Use the supplied code "
                    "definitions when discussing a "
                    "catalogue classification. If no "
                    "catalogue record is supplied, "
                    "describe the check "
                    "conditionally.",
                    "Tie each proposed check to a "
                    "specific structural feature and "
                    "explain what result would "
                    "support or challenge the "
                    "classification.",
                    "Separate possible "
                    "classification or observation "
                    "differences from geological "
                    "explanations. Agreement alone "
                    "does not establish a formation "
                    "mechanism.",
                ],
            },
            {
                "question_id": "Q2",
                "question_type": "GENERAL_CRATER_INVESTIGATION",
                "text": "For {crater}, is any apparent asymmetry more "
                "consistent with variation within the crater or "
                "with the surrounding terrain? Assess the "
                "evidence, propose analyses that could "
                "distinguish these explanations, and identify "
                "the observations that would change your "
                "interpretation.",
                "answer_requirements": [
                    "Establish whether the supplied "
                    "evidence shows an asymmetry "
                    "before explaining it; an "
                    "inconclusive assessment is "
                    "acceptable.",
                    "Relate comparisons to their "
                    "locations and directions. "
                    "Consider how the catalogue "
                    "centre and sampled extent "
                    "affect the distinction between "
                    "crater relief and background "
                    "slope.",
                    "State which comparisons the "
                    "current representations support "
                    "and what any further "
                    "measurement would require. Keep "
                    "proposed measurements separate "
                    "from results already "
                    "obtained.",
                ],
            },
            {
                "question_id": "Q3",
                "question_type": "CROSS_MODAL_RELATIONSHIP_DISCOVERY",
                "text": "For {crater}, what evidence would distinguish a "
                "difference in surface material between the "
                "central area and its surroundings from effects "
                "of landform, illumination or spatial averaging? "
                "Assess any supplied regional contrasts and "
                "propose analyses that could separate the "
                "plausible explanations.",
                "answer_requirements": [
                    "If regional spectral values are "
                    "supplied, ground the discussion "
                    "in a specific "
                    "region-and-wavelength "
                    "comparison. Otherwise, describe "
                    "the spectral comparison as a "
                    "proposed investigation.",
                    "Use any available structural or "
                    "elevation evidence to assess "
                    "the explanations. Distinguish "
                    "geometric sampling regions from "
                    "mapped geological units.",
                    "Explain what would favour or "
                    "weaken a material-difference "
                    "interpretation. Account for "
                    "spatial averaging and "
                    "uncertainty without assigning a "
                    "unique composition from these "
                    "inputs.",
                ],
            },
            {
                "question_id": "Q4",
                "question_type": "EXPLANATORY_PERSPECTIVE_EXPLORATION",
                "text": "What evidence, if any, suggests that the "
                "interior of {crater} was modified after its "
                "initial formation? Develop an interpretation "
                "from the supplied features, consider plausible "
                "alternatives, and propose analyses whose "
                "outcomes would help distinguish them.",
                "answer_requirements": [
                    "Anchor explanations to specific "
                    "supplied features or values. If "
                    "the evidence is insufficient, "
                    "keep the interpretation "
                    "conditional and identify the "
                    "unresolved issue.",
                    "Explain how an analysis could "
                    "favour one mechanism over "
                    "another. Include preservation "
                    "of the original structure as an "
                    "alternative where relevant.",
                    "Prioritize comparisons possible "
                    "with the supplied packages. "
                    "State additional requirements "
                    "for other proposed analyses, "
                    "and distinguish proposed work "
                    "from observed or computed "
                    "results.",
                ],
            },
        ],
    },
}
DEVELOPMENT_ALIGNMENT = DEVELOPMENT_SETTINGS["alignment"]
DEVELOPMENT_COMPLEMENTARITY = DEVELOPMENT_SETTINGS["complementarity"]
DEVELOPMENT_QUESTIONS = DEVELOPMENT_SETTINGS["questions"]

## Shared generation interface

In [ ]:
import copy
import time
from contextlib import ExitStack
from typing import Any, Literal
from unittest.mock import patch

import torch
import transformers
from pydantic import BaseModel, ConfigDict
from torch.nn.attention import SDPBackend, sdpa_kernel


class GenerationTrace(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        strict=True,
        allow_inf_nan=False,
    )

    messages: list[dict[str, Any]]
    max_new_tokens: int
    input_tokens: int | None = None
    output_tokens: int | None = None
    finish_reason: Literal["error", "eos", "length", "unknown"] = "error"
    elapsed_seconds: float = 0.0
    peak_gpu_gib: float | None = None
    text: str | None = None
    error: str | None = None


GENERATION_LOG: list[dict[str, Any]] = []


def generate_reply(
    messages: list[dict],
    max_new_tokens: int = 128,
    repetition_penalty: float = 1.0,
) -> str:
    """Generate with model-specific attention and preserve outcome logs."""
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be positive.")

    trace = GenerationTrace(
        messages=copy.deepcopy(messages),
        max_new_tokens=max_new_tokens,
    )
    record = trace.model_dump()
    GENERATION_LOG.append(record)

    inputs = outputs = new_tokens = None
    started = time.perf_counter()
    peak_tracking_started = False

    try:
        if MODEL_NAME not in MODEL_CLASSES:
            raise RuntimeError(f"Unsupported model: {MODEL_NAME}")
        if MODEL_NAME != MODEL_ID or str(model.config._name_or_path) != MODEL_NAME:
            raise RuntimeError("Model selection changed. Use a fresh runtime.")
        is_qwen = MODEL_NAME == "Qwen/Qwen3-VL-8B-Instruct"
        if is_qwen:
            if transformers.__version__ != "4.57.6":
                raise RuntimeError("The Qwen adapter requires Transformers 4.57.6.")
            import transformers.integrations.sdpa_attention as hf_sdpa

            if not callable(getattr(hf_sdpa, "use_gqa_in_sdpa", None)):
                raise RuntimeError("Expected Qwen SDPA helper is unavailable.")

        if not torch.cuda.is_available():
            raise RuntimeError("A CUDA GPU is required.")

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        peak_tracking_started = True

        # Preprocess and check input length before moving tensors to the GPU.
        template_kwargs = {} if is_qwen else {"enable_thinking": False}
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            **template_kwargs,
        )

        n_input = int(inputs["input_ids"].shape[-1])
        trace.input_tokens = n_input

        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(f"INPUT_TOKEN_LIMIT: {n_input} > {MAX_INPUT_TOKENS}")

        has_images = any(
            block.get("type") == "image" for message in messages for block in message["content"]
        )
        if has_images and not any("pixel" in name for name in inputs):
            raise ValueError("Processor output has no images; multimodal generation is blocked.")

        inputs = inputs.to("cuda:0", dtype=model.dtype)

        from transformers import GenerationConfig

        config = copy.deepcopy(model.generation_config)
        defaults = GenerationConfig()

        # Reset sampling-only settings for deterministic greedy decoding.
        for name in (
            "temperature",
            "top_p",
            "top_k",
            "min_p",
            "typical_p",
            "epsilon_cutoff",
            "eta_cutoff",
            "top_h",
        ):
            if hasattr(config, name) and hasattr(defaults, name):
                setattr(config, name, getattr(defaults, name))

        config.do_sample = False
        config.num_beams = 1
        config.num_return_sequences = 1
        config.max_new_tokens = max_new_tokens
        config.repetition_penalty = repetition_penalty
        config.use_cache = True
        config.return_dict_in_generate = False
        config.output_scores = False
        config.output_logits = False
        config.output_attentions = False
        config.output_hidden_states = False

        # Retain this argument only for the pinned Qwen/Transformers 4.57.6 branch.
        generation_kwargs = {"use_model_defaults": False} if is_qwen else {}

        with torch.inference_mode(), ExitStack() as stack:
            if is_qwen:
                stack.enter_context(
                    patch.object(
                        hf_sdpa,
                        "use_gqa_in_sdpa",
                        new=lambda attention_mask, key: False,
                    )
                )
                stack.enter_context(sdpa_kernel(backends=[SDPBackend.EFFICIENT_ATTENTION]))

            outputs = model.generate(
                **inputs,
                generation_config=config,
                **generation_kwargs,
            )

        new_tokens = outputs[:, n_input:]
        token_ids = new_tokens[0].tolist()

        trace.text = processor.batch_decode(
            new_tokens,
            skip_special_tokens=True,
        )[0]
        trace.output_tokens = len(token_ids)

        eos = config.eos_token_id
        if eos is None:
            eos = processor.tokenizer.eos_token_id

        eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])

        if token_ids and token_ids[-1] in eos_ids:
            trace.finish_reason = "eos"
        elif len(token_ids) >= max_new_tokens:
            trace.finish_reason = "length"
        else:
            trace.finish_reason = "unknown"

        torch.cuda.synchronize()
        return trace.text

    except Exception as exc:
        trace.finish_reason = "error"
        trace.error = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        trace.elapsed_seconds = time.perf_counter() - started

        if peak_tracking_started:
            trace.peak_gpu_gib = torch.cuda.max_memory_allocated() / 1024**3

        record.update(trace.model_dump())
        del inputs, outputs, new_tokens

## Embedded target interfaces
Definitions only; no model calls.

In [ ]:
EXECUTION_SOURCES = {
    "target_setup": "import hashlib\n"
    "import importlib.metadata\n"
    "import json\n"
    "import subprocess\n"
    "import sys\n"
    "from pathlib import Path, PureWindowsPath\n"
    "\n"
    "import torch\n"
    "from pydantic import BaseModel, ConfigDict\n"
    "\n"
    'PROJECT = Path("/content/autonomous-modality-selection")\n'
    'CASE_DIR = DATASET_ROOT / f"crater-{TARGET_ID}"\n'
    "\n"
    "\n"
    "if not IS_DEVELOPMENT or BUDGET not in (3, 4):\n"
    '    raise ValueError("This notebook is development-only; budget must '
    'be 3 or 4.")\n'
    'sys.path.insert(0, str(PROJECT / "src"))\n'
    "sys.path.insert(0, str(PROJECT / "
    '"extensions/evidence_views_v1/src"))\n'
    "\n"
    "from autonomous_modality.development_inputs import (\n"
    "    ANSWER_POLICY,\n"
    "    ANSWER_POLICY_VERSION,\n"
    ")\n"
    "from autonomous_modality.experiments import scenario_seed\n"
    "from autonomous_modality.pilot import CatalogueRow\n"
    "\n"
    "MODEL_NAME = str(model.config._name_or_path)\n"
    'ALLOWED_MODELS = ("Qwen/Qwen3-VL-8B-Instruct", '
    '"google/gemma-4-E4B-it")\n'
    "if MODEL_NAME not in ALLOWED_MODELS:\n"
    '    raise ValueError(f"Loaded model is outside this experiment: '
    '{MODEL_NAME}")\n'
    'MODEL_TAG = "qwen3-vl-8b" if ALLOWED_MODELS[0] == MODEL_NAME else '
    '"gemma4-e4b"\n'
    "OUTPUT = RESULTS_ROOT / RUN_LABEL / MODEL_TAG / "
    'f"crater-{TARGET_ID}" / f"budget-{BUDGET}"\n'
    "\n"
    "\n"
    "def sha256(path: Path) -> str:\n"
    '    """Hash files in chunks to limit memory use."""\n'
    "    digest = hashlib.sha256()\n"
    '    with path.open("rb") as stream:\n'
    '        for block in iter(lambda: stream.read(1024 * 1024), b""):\n'
    "            digest.update(block)\n"
    "    return digest.hexdigest()\n"
    "\n"
    "\n"
    "def json_digest(value: object) -> str:\n"
    '    """Compare JSON configurations while rejecting NaN."""\n'
    "    raw = json.dumps(\n"
    '        value, ensure_ascii=False, sort_keys=True, separators=(",", '
    '":"), allow_nan=False\n'
    "    )\n"
    '    return hashlib.sha256(raw.encode("utf-8")).hexdigest()\n'
    "\n"
    "\n"
    "def write_new_json(path: Path, value: object) -> None:\n"
    '    """Create a new file without overwriting existing results."""\n'
    '    with path.open("x", encoding="utf-8") as stream:\n'
    "        json.dump(value, stream, ensure_ascii=False, indent=2, "
    "allow_nan=False)\n"
    "\n"
    "\n"
    "def portable_name(value: str) -> str:\n"
    '    """Support Windows paths in records uploaded to Colab."""\n'
    "    return PureWindowsPath(value).name\n"
    "\n"
    "\n"
    "from autonomous_modality.colab_four_modality import "
    "prepare_compact_inputs\n"
    "from autonomous_modality.multispectral_dataset import "
    "FourModalityManifest\n"
    "from autonomous_modality.multispectral_release import "
    "verify_release\n"
    "\n"
    "release = verify_release(DATASET_ROOT)\n"
    "if TARGET_ID not in {entry.case_id for entry in release.cases}:\n"
    '    raise ValueError("Target is not in the verified release.")\n'
    "package = FourModalityManifest.model_validate_json((CASE_DIR / "
    '"manifest.json").read_bytes())\n'
    "catalogue = CatalogueRow.model_validate_json((CASE_DIR / "
    '"core_views/catalogue.json").read_bytes())\n'
    "if catalogue.id != TARGET_ID or package.case_id != TARGET_ID:\n"
    '    raise ValueError("Target identity mismatch.")\n'
    "CASE_ID, CASE_NAME = str(catalogue.id), catalogue.name\n"
    "SOURCES = {\n"
    '    "catalogue.json": CASE_DIR / "core_views/catalogue.json",\n'
    '    "manifest.json": CASE_DIR / "manifest.json",\n'
    '    "summary.json": CASE_DIR / "summary.json",\n'
    '    "multispectral.npz": CASE_DIR / "multispectral.npz",\n'
    '    "release.json": DATASET_ROOT / "release.json",\n'
    "}\n"
    "SOURCE_HASHES = {name: sha256(path) for name, path in "
    "SOURCES.items()}\n"
    "question_path = RESULTS_ROOT / RUN_LABEL / "
    '"development_questions.json"\n'
    "question_path.parent.mkdir(parents=True, exist_ok=True)\n"
    "if question_path.exists():\n"
    "    if json.loads(question_path.read_text()) != "
    "DEVELOPMENT_QUESTIONS:\n"
    '        raise ValueError("Questions changed; use a new RUN_LABEL.")\n'
    "else:\n"
    "    write_new_json(question_path, DEVELOPMENT_QUESTIONS)\n"
    "QUESTIONS = "
    'json.loads(question_path.read_text(encoding="utf-8"))["questions"]\n'
    'assert [q["question_id"] for q in QUESTIONS] == ["Q1", "Q2", "Q3", '
    '"Q4"]\n'
    "\n"
    "\n"
    "class ViewReceipt(BaseModel):\n"
    '    model_config = ConfigDict(extra="forbid", strict=True)\n'
    "    version: str\n"
    "    source_sha256: dict[str, str]\n"
    "    implementation_sha256: dict[str, str]\n"
    "    image_edge: int\n"
    "    output_sha256: dict[str, str]\n"
    "\n"
    "\n"
    "view_code = PROJECT / "
    '"extensions/evidence_views_v1/src/ams_evidence_views"\n'
    "CODE_HASHES = {\n"
    "    str(p.relative_to(PROJECT)): sha256(p)\n"
    '    for root in (PROJECT / "src/autonomous_modality", view_code)\n'
    '    for p in sorted(root.glob("*.py"))\n'
    "}\n"
    'VIEW_DIR = RESULTS_ROOT / RUN_LABEL / "shared_inputs" / CASE_ID\n'
    "receipt = "
    "ViewReceipt.model_validate(prepare_compact_inputs(CASE_DIR, "
    "VIEW_DIR, IMAGE_EDGE))\n"
    'receipt_path = VIEW_DIR / "receipt.json"\n'
    "PREVIEWS = {\n"
    '    key: VIEW_DIR / f"{key}.png"\n'
    "    for key in (\n"
    '        "optical_local",\n'
    '        "optical_context",\n'
    '        "elevation",\n'
    '        "profile_east",\n'
    '        "profile_north",\n'
    "    )\n"
    "}\n"
    "TERRAIN_TEXT = (VIEW_DIR / "
    '"terrain.txt").read_text(encoding="utf-8")\n'
    "MULTISPECTRAL_TEXT = (VIEW_DIR / "
    '"multispectral.txt").read_text(encoding="utf-8")\n'
    "CATALOGUE_TEXT = json.dumps(catalogue.model_dump(), "
    "ensure_ascii=False)\n"
    "OUTPUT.mkdir(parents=True, exist_ok=True)\n"
    "from autonomous_modality.modality_descriptions import (\n"
    "    ASSET_DESCRIPTION_VERSION,\n"
    "    describe_assets,\n"
    "    validate_asset_descriptions,\n"
    ")\n"
    "\n"
    "INVENTORY_VERSION = ASSET_DESCRIPTION_VERSION\n"
    "ASSETS = describe_assets(\n"
    "    {\n"
    '        "CRATER_CATALOG": {"cost": 1},\n'
    '        "OPTICAL_IMAGE": {"cost": 2},\n'
    '        "TOPOGRAPHY": {"cost": 2},\n'
    '        "MULTISPECTRAL_IMAGE": {"cost": 2},\n'
    "    }\n"
    ")\n"
    "validate_asset_descriptions(ASSETS)\n"
    "from autonomous_modality.development_inputs import "
    "SELECTOR_ALIGNMENT, SELECTOR_ALIGNMENT_VERSION\n"
    "\n"
    'if SELECTOR_VARIANT == "updated":\n'
    "    SELECTOR_ALIGNMENT = DEVELOPMENT_ALIGNMENT\n"
    '    SELECTOR_ALIGNMENT_VERSION = "selector-qualitative-dev-2.0"\n'
    "\n"
    "SELECTOR_TASK = (\n"
    '    "Choose one supplied feasible input combination expected to '
    'support distinct, "\n'
    '    "relevant analytical approaches and explanatory perspectives. " '
    "+ SELECTOR_ALIGNMENT\n"
    ")\n"
    "# Shared, versioned answer policy imported above; all five "
    "conditions use it.\n"
    "CONFIG = {\n"
    '    "protocol": RECORD_VERSION,\n'
    '    "development_revision": "second-step-sufficiency-1.0",\n'
    '    "second_step_review_enabled": SECOND_STEP_REVIEW_ENABLED,\n'
    '    "second_step_review_version": "observation-sufficiency-1.0",\n'
    '    "second_step_review_prompt": SECOND_STEP_REVIEW_PROMPT,\n'
    '    "second_step_review_trigger": '
    '"one_acquired_modality_with_observation",\n'
    '    "selector_variant": SELECTOR_VARIANT,\n'
    '    "selector_alignment_version": SELECTOR_ALIGNMENT_VERSION,\n'
    '    "iterative_prompt_extension": (DEVELOPMENT_ALIGNMENT + " " + '
    "DEVELOPMENT_COMPLEMENTARITY)\n"
    '        if SELECTOR_VARIANT == "updated" else "",\n'
    '    "execution_sources_sha256": EXECUTION_SOURCES_SHA256,\n'
    '    "resolved_model_revision": RESOLVED_MODEL_REVISION,\n'
    '    "loaded_attention_config": LOADED_ATTENTION_CONFIG,\n'
    '    "split": "development",\n'
    '    "case_id": CASE_ID,\n'
    '    "case_name": CASE_NAME,\n'
    '    "model": MODEL_NAME,\n'
    '    "attention_policy": ATTENTION_POLICY,\n'
    '    "model_revision": getattr(model.config, "_commit_hash", None),\n'
    '    "four_bit": bool(getattr(model, "is_loaded_in_4bit", False)),\n'
    '    "seed": SEED,\n'
    '    "budget": BUDGET,\n'
    '    "maximum_modalities": MAX_MODALITIES,\n'
    '    "random_draws": RANDOM_DRAWS,\n'
    '    "image_edge": IMAGE_EDGE,\n'
    '    "answer_tokens": ANSWER_TOKENS,\n'
    '    "answer_repetition_penalty": ANSWER_REPETITION_PENALTY,\n'
    '    "dataset_release_sha256": sha256(DATASET_ROOT / '
    '"release.json"),\n'
    '    "development_extension": "four-modality-1.0",\n'
    '    "selector_tokens": SELECTOR_TOKENS,\n'
    '    "max_input_tokens": MAX_INPUT_TOKENS,\n'
    '    "decoding": "greedy_one_beam",\n'
    '    "thinking": "disabled",\n'
    '    "language": "en",\n'
    '    "language_protocol": "english-ap-1.0",\n'
    '    "answer_policy_version": ANSWER_POLICY_VERSION,\n'
    '    "inventory_version": ASSET_DESCRIPTION_VERSION,\n'
    '    "asset_description_version": ASSET_DESCRIPTION_VERSION,\n'
    '    "asset_description_scope": "all_selectors",\n'
    '    "cost_unit": "legacy_ordinal_units",\n'
    '    "iterative_selector_tokens": ITERATIVE_SELECTOR_TOKENS,\n'
    '    "question_sha256": sha256(question_path),\n'
    '    "source_sha256": SOURCE_HASHES,\n'
    '    "view_receipt": receipt.model_dump(),\n'
    '    "selector_task": SELECTOR_TASK,\n'
    '    "answer_policy": ANSWER_POLICY,\n'
    '    "assets": ASSETS,\n'
    '    "repository_commit": subprocess.check_output(\n'
    '        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True\n'
    "    ).strip(),\n"
    '    "implementation_sha256": CODE_HASHES,\n'
    "}\n"
    'print("Development case:", CASE_NAME, "| Model:", MODEL_NAME, "| '
    'Output:", OUTPUT)\n'
    'print("Per model: 4 questions x (1 + 1 + 5 + 1 + 1) = 36 development '
    'records.")',
    "selector": "import random\n"
    "import re\n"
    "from itertools import combinations\n"
    "\n"
    "from pydantic import BaseModel, ConfigDict, Field\n"
    "\n"
    "\n"
    "class AgentChoice(BaseModel):\n"
    '    model_config = ConfigDict(extra="forbid", strict=True)\n'
    "    option_id: str\n"
    "    rationale: str = Field(min_length=1)\n"
    "\n"
    "\n"
    "def parse_agent_choice(raw: str) -> AgentChoice:\n"
    '    """Parse one-shot-json-envelope-v1: bare JSON or one complete JSON '
    "fence.\n"
    "\n"
    "    Original output stays in GENERATION_LOG. No repairs or generation "
    "retries occur.\n"
    '    """\n'
    "    text = raw.strip()\n"
    '    if text.startswith("```"):\n'
    '        match = re.fullmatch(r"```(?:json)?[ '
    '\\t]*\\r?\\n([\\s\\S]*?)\\r?\\n```", text)\n'
    '        if match is None or "```" in match.group(1):\n'
    '            raise ValueError("INVALID_AGENT_JSON_ENVELOPE")\n'
    "        text = match.group(1).strip()\n"
    "    return AgentChoice.model_validate_json(text)\n"
    "\n"
    "\n"
    "def valid_selection(selected: object) -> list[str]:\n"
    '    """Validate RANDOM/AGENT selections; ALL_AVAILABLE has no '
    'selection-budget cap."""\n'
    "    if not isinstance(selected, list) or not all(isinstance(x, str) for "
    "x in selected):\n"
    '        raise ValueError("Selection must be a list of strings.")\n'
    "    if not 1 <= len(selected) <= MAX_MODALITIES or len(set(selected)) != "
    "len(selected):\n"
    '        raise ValueError("Invalid selection count or duplicate '
    'modalities.")\n'
    "    if set(selected) - set(ASSETS):\n"
    '        raise ValueError("Unknown modality.")\n'
    '    if sum(ASSETS[x]["cost"] for x in selected) > BUDGET:\n'
    '        raise ValueError("Selection exceeds the budget.")\n'
    "    return sorted(selected)\n"
    "\n"
    "\n"
    "FEASIBLE = [\n"
    "    list(items)\n"
    "    for n in range(1, MAX_MODALITIES + 1)\n"
    "    for items in combinations(sorted(ASSETS), n)\n"
    '    if sum(ASSETS[x]["cost"] for x in items) <= BUDGET\n'
    "]\n"
    "OPTIONS = {\n"
    '    f"S{i + 1}": {"selected": items, "total_cost": sum(ASSETS[x]["cost"] '
    "for x in items)}\n"
    "    for i, items in enumerate(FEASIBLE)\n"
    "}\n"
    "if not OPTIONS:\n"
    '    raise ValueError("NO_FEASIBLE_OPTIONS")\n'
    "\n"
    "\n"
    "def select_inputs(\n"
    "    condition: str, question: str, question_id: str, draw_index: int = "
    "0\n"
    ") -> tuple[list[str], str]:\n"
    '    """Retain three positional arguments and optional draw_index, '
    'without saved aliases."""\n'
    "    if type(draw_index) is not int or not 0 <= draw_index < "
    "RANDOM_DRAWS:\n"
    '        raise ValueError("Invalid draw_index.")\n'
    '    if condition != "RANDOM" and draw_index != 0:\n'
    '        raise ValueError("Non-random conditions have one attempt.")\n'
    '    if condition == "NO_DATA":\n'
    '        return [], "No additional scientific evidence."\n'
    '    if condition == "ALL_AVAILABLE":\n'
    '        return sorted(ASSETS), "All available modalities; '
    'non-budget-matched reference."\n'
    '    if condition == "RANDOM":\n'
    '        seed = scenario_seed(SEED, f"{CASE_ID}-{question_id}", BUDGET, '
    "draw_index)\n"
    "        return valid_selection(random.Random(seed).choice(FEASIBLE)), (\n"
    '            "Uniform feasible-subset draw with replacement."\n'
    "        )\n"
    '    if condition != "AGENT":\n'
    '        raise ValueError(f"Unknown condition: {condition}")\n'
    "\n"
    "    instruction = {\n"
    '        "task": SELECTOR_TASK,\n'
    '        "alignment_version": SELECTOR_ALIGNMENT_VERSION,\n'
    '        "answer_requirements": next(\n'
    '            q["answer_requirements"] for q in QUESTIONS if '
    'q["question_id"] == question_id\n'
    "        ),\n"
    '        "question": question,\n'
    '        "modality_descriptions": ASSETS,\n'
    '        "feasible_options": OPTIONS,\n'
    '        "rules": [\n'
    '            "Choose exactly one supplied option_id; do not change the '
    'option.",\n'
    '            "Budget is a ceiling; spending less receives no separate '
    'reward.",\n'
    '            "The catalog is one historical target record, not population '
    'data.",\n'
    '            "Check content_inventory for actual and absent fields; '
    'unlisted fields are unknown.",\n'
    '            "Inputs include plotted views, terrain samples and regional '
    'multispectral statistics, "\n'
    '            "not executable rasters.",\n'
    '            "Acknowledge representation limitations; "\n'
    '            "focus on analytical approaches and explanations.",\n'
    '            "Return only JSON with exactly option_id and a nonempty '
    'rationale.",\n'
    '            "Write the rationale entirely in English.",\n'
    '            "The rationale should be less than 200 characters.",\n'
    "        ],\n"
    "    }\n"
    "    raw = generate_reply(\n"
    "        [\n"
    "            {\n"
    '                "role": "user",\n'
    '                "content": [{"type": "text", "text": '
    "json.dumps(instruction, ensure_ascii=False)}],\n"
    "            }\n"
    "        ],\n"
    "        max_new_tokens=SELECTOR_TOKENS,\n"
    "    )\n"
    "    # GENERATION_LOG retains raw output for both successful and failed "
    "attempts.\n"
    '    if GENERATION_LOG[-1]["finish_reason"] != "eos":\n'
    '        raise ValueError("SELECTOR_OUTPUT_NOT_COMPLETE")\n'
    "    parsed = parse_agent_choice(raw)\n"
    "    if parsed.option_id not in OPTIONS or not parsed.rationale.strip():\n"
    '        raise ValueError("INVALID_AGENT_CHOICE")\n'
    '    return valid_selection(OPTIONS[parsed.option_id]["selected"]), '
    "parsed.rationale\n"
    "\n"
    "\n"
    "assert len(FEASIBLE) == (10 if BUDGET == 4 else 7)\n"
    'assert select_inputs("NO_DATA", "test", "Q1") == ([], "No additional '
    'scientific evidence.")\n'
    'print("Feasible combinations:", OPTIONS)\n'
    "print(\n"
    '    "Five random draws for Q1 (repeated combinations are valid):",\n'
    '    [select_inputs("RANDOM", "test", "Q1", i)[0] for i in '
    "range(RANDOM_DRAWS)],\n"
    ")",
    "answer": "def build_answer_messages(\n"
    "    question: str, requirements: list[str], selected: list[str]\n"
    ") -> list[dict]:\n"
    '    """Route only selected evidence; exclude condition and selector '
    'metadata."""\n'
    "    if len(set(selected)) != len(selected) or set(selected) - "
    "set(ASSETS):\n"
    '        raise ValueError("Invalid evidence selection.")\n'
    "    prompt = (\n"
    "        ANSWER_POLICY\n"
    '        + f"\\nTarget: '
    '{CASE_NAME}\\nQuestion:\\n{question}\\nRequirements:\\n"\n'
    '        + "\\n".join(f"- {x}" for x in requirements)\n'
    "    )\n"
    "    images, texts = [], []\n"
    '    if "CRATER_CATALOG" in selected:\n'
    "        texts.append(\n"
    '            "Historical catalogue record; morphology codes are not ground '
    'truth. "\n'
    '            "Coordinates: degrees; diameter: km. "\n'
    '            "int_shp: b=bowl, sh=slump hummocks, ff=flat floor; "\n'
    '            "rim_shp: c=circular, sc=scalloped, t=terraced; "\n'
    '            "cent_struc: cp=central peak, mp=multiple peaks, pi=central '
    'pit, "\n'
    '            "pr=peak ring, mr=multi-ring, n=none; x=poorly imaged; "\n'
    '            "rayed: y=yes, n=no. Unknown codes must not be guessed.\\n" + '
    "CATALOGUE_TEXT\n"
    "        )\n"
    '    if "OPTICAL_IMAGE" in selected:\n'
    '        for key in ("optical_local", "optical_context"):\n'
    '            images.append({"type": "image", "image": str(PREVIEWS[key])})\n'
    '            texts.append(f"Image {len(images)}: {key}; display-stretched '
    'optical view.")\n'
    '    if "TOPOGRAPHY" in selected:\n'
    '        for key in ("elevation", "profile_east", "profile_north"):\n'
    '            images.append({"type": "image", "image": str(PREVIEWS[key])})\n'
    '            texts.append(f"Image {len(images)}: {key}; elevation in '
    'metres, not crater depth.")\n'
    "        texts.append(TERRAIN_TEXT)\n"
    '    if "MULTISPECTRAL_IMAGE" in selected:\n'
    "        texts.append(MULTISPECTRAL_TEXT)\n"
    "    if not selected:\n"
    '        texts.append("No additional scientific evidence is supplied.")\n'
    "    # Both models receive the same content and order: images followed by "
    "text.\n"
    '    content = [*images, {"type": "text", "text": prompt + "\\n\\n" + '
    '"\\n\\n".join(texts)}]\n'
    '    return [{"role": "user", "content": content}]\n'
    "\n"
    "\n"
    "def answer_question(question: str, requirements: list[str], selected: "
    "list[str]) -> str:\n"
    '    """Call the loaded model with the token cap recorded in the run '
    'configuration."""\n'
    "    return generate_reply(\n"
    "        build_answer_messages(question, requirements, selected),\n"
    "        max_new_tokens=ANSWER_TOKENS,\n"
    "        repetition_penalty=ANSWER_REPETITION_PENALTY,\n"
    "    )\n"
    "\n"
    "\n"
    "# Check routing without calling the model.\n"
    "for chosen, expected in [\n"
    "    ([], 0),\n"
    '    (["CRATER_CATALOG"], 0),\n'
    '    (["OPTICAL_IMAGE"], 2),\n'
    '    (["TOPOGRAPHY"], 3),\n'
    '    (["MULTISPECTRAL_IMAGE"], 0),\n'
    "    (sorted(ASSETS), 5),\n"
    "]:\n"
    '    content = build_answer_messages("fixture", [], chosen)[0]["content"]\n'
    '    assert sum(x["type"] == "image" for x in content) == expected\n'
    'print("Input routing verified: 0/0/2/3/5 images.")',
    "iterative": "def selector_generate_reply(messages: list[dict], max_new_tokens: int = "
    "512) -> str:\n"
    '    """Add the reminder only after one verified acquisition; log '
    'effective messages."""\n'
    '    if SELECTOR_VARIANT != "previous":\n'
    '        raise ValueError("The second-step ablation requires the '
    'previous selector.")\n'
    "    actual = copy.deepcopy(messages)\n"
    "    if SECOND_STEP_REVIEW_ENABLED:\n"
    '        if len(actual) != 2 or actual[0].get("role") != "system" or '
    'actual[1].get("role") != "user":\n'
    '            raise ValueError("Unexpected iterative message '
    'structure.")\n'
    '        payload = json.loads(actual[1]["content"][0]["text"])\n'
    '        accessed = payload.get("accessed_modalities")\n'
    "        if not isinstance(accessed, list) or any(not isinstance(m, str) "
    "for m in accessed):\n"
    '            raise ValueError("Invalid acquisition state.")\n'
    "        if len(accessed) > 1:\n"
    '            raise ValueError("Unexpected selector call after the '
    'acquisition limit.")\n'
    "        if len(accessed) == 1:\n"
    '            if not any(b.get("type") == "text" and b.get("text", '
    '"").startswith(\n'
    '                "Acquired evidence: "\n'
    '            ) for b in actual[1]["content"][1:]):\n'
    '                raise ValueError("Acquired evidence is missing from the '
    'observation context.")\n'
    '            actual[0]["content"].append({"type": "text", "text": '
    "SECOND_STEP_REVIEW_PROMPT})\n"
    "    return generate_reply(actual, max_new_tokens=max_new_tokens)\n"
    "\n"
    "\n"
    "from autonomous_modality.iterative import (\n"
    "    ContentBlock,\n"
    "    EvidenceObservation,\n"
    "    IterativePolicy,\n"
    "    IterativeSelection,\n"
    "    run_iterative_selection,\n"
    ")\n"
    "from autonomous_modality.iterative_colab import notebook_generator\n"
    "from autonomous_modality.models import (\n"
    "    CraterQuestion,\n"
    "    CraterReference,\n"
    "    DataAssetProfile,\n"
    "    InputDataModality,\n"
    "    InputSelectionConstraints,\n"
    "    InputSelectionRequest,\n"
    ")\n"
    "\n"
    "ITERATIVE_POLICY = IterativePolicy(\n"
    '    cost_unit="legacy_ordinal_units",\n'
    "    cost_definition=(\n"
    '        "Ordinal costs: catalogue=1, optical=2, topography=2, '
    'multispectral=2; complete packages."\n'
    "    ),\n"
    "    selector_max_new_tokens=ITERATIVE_SELECTOR_TOKENS,\n"
    ")\n"
    "PINNED_RECEIPT_SHA256 = sha256(receipt_path)\n"
    "\n"
    "\n"
    "def build_evidence_content(selected: list[str]) -> list[dict]:\n"
    '    """Reuse the original routing, removing only the shared answer '
    'instruction prefix."""\n'
    '    question = "Evidence routing placeholder"\n'
    "    prefix = (\n"
    '        ANSWER_POLICY + f"\\nTarget: '
    '{CASE_NAME}\\nQuestion:\\n{question}\\nRequirements:\\n" + "\\n\\n"\n'
    "    )\n"
    "    content = copy.deepcopy(build_answer_messages(question, [], "
    'selected)[0]["content"])\n'
    '    if content[-1]["type"] != "text" or not '
    'content[-1]["text"].startswith(prefix):\n'
    '        raise ValueError("Answer routing changed; verify evidence-only '
    'extraction.")\n'
    '    content[-1]["text"] = content[-1]["text"][len(prefix) :]\n'
    "    return content\n"
    "\n"
    "\n"
    "def load_notebook_evidence(modality: InputDataModality) -> "
    "EvidenceObservation:\n"
    '    """Return actual receipt-verified input bytes through the original '
    'notebook routing."""\n'
    "    if sha256(receipt_path) != PINNED_RECEIPT_SHA256:\n"
    '        raise ValueError("Input receipt changed during acquisition.")\n'
    "    for name, digest in receipt.output_sha256.items():\n"
    "        if sha256(VIEW_DIR / name) != digest:\n"
    '            raise ValueError(f"Input view changed: {name}")\n'
    '    if sha256(SOURCES["catalogue.json"]) != '
    'SOURCE_HASHES["catalogue.json"]:\n'
    '        raise ValueError("Catalogue changed during acquisition.")\n'
    "    actual_catalogue = CatalogueRow.model_validate_json(\n"
    '        SOURCES["catalogue.json"].read_text(encoding="utf-8")\n'
    "    )\n"
    "    if json.dumps(actual_catalogue.model_dump(), ensure_ascii=False) != "
    "CATALOGUE_TEXT:\n"
    '        raise ValueError("In-memory catalogue differs from pinned '
    'input.")\n'
    '    if (VIEW_DIR / "terrain.txt").read_text(encoding="utf-8") != '
    "TERRAIN_TEXT:\n"
    '        raise ValueError("In-memory terrain differs from pinned '
    'input.")\n'
    '    if (VIEW_DIR / "multispectral.txt").read_text(encoding="utf-8") != '
    "MULTISPECTRAL_TEXT:\n"
    '        raise ValueError("In-memory multispectral evidence changed.")\n'
    "    files_by_modality = {\n"
    '        "MULTISPECTRAL_IMAGE": {"multispectral.txt": '
    'receipt.output_sha256["multispectral.txt"]},\n'
    '        "CRATER_CATALOG": {"catalogue.json": '
    'SOURCE_HASHES["catalogue.json"]},\n'
    '        "OPTICAL_IMAGE": {\n'
    "            name: receipt.output_sha256[name]\n"
    '            for name in ("optical_local.png", "optical_context.png")\n'
    "        },\n"
    '        "TOPOGRAPHY": {\n'
    "            name: receipt.output_sha256[name]\n"
    '            for name in ("elevation.png", "profile_east.png", '
    '"profile_north.png", "terrain.txt")\n'
    "        },\n"
    "    }\n"
    "    return EvidenceObservation(\n"
    "        modality=modality,\n"
    "        blocks=[ContentBlock.model_validate(b) for b in "
    "build_evidence_content([modality.value])],\n"
    "        evidence_sha256=files_by_modality[modality.value],\n"
    "        package_sha256=PINNED_RECEIPT_SHA256,\n"
    "    )\n"
    "\n"
    "\n"
    "def select_inputs_iterative(question: str, question_id: str) -> "
    "IterativeSelection:\n"
    '    """A separate entry point; the original select_inputs function '
    'remains unchanged."""\n'
    '    spec = next(q for q in QUESTIONS if q["question_id"] == '
    "question_id)\n"
    "    request = InputSelectionRequest(\n"
    "        question=CraterQuestion(\n"
    '            question_id=f"{CASE_ID}-{question_id}",\n'
    "            text=question,\n"
    '            question_type=spec["question_type"],\n'
    '            answer_requirements=spec["answer_requirements"],\n'
    "            crater=CraterReference(\n"
    "                crater_id=CASE_ID,\n"
    "                name=CASE_NAME,\n"
    "                latitude=catalogue.lat_n,\n"
    "                longitude=catalogue.lon_e_0,\n"
    "            ),\n"
    "        ),\n"
    "        assets=[\n"
    "            DataAssetProfile(\n"
    "                asset_id=name,\n"
    "                modality=name,\n"
    "                title=name,\n"
    '                estimated_cost=asset["cost"],\n'
    '                limitations=[asset["description"]],\n'
    '                content_inventory=asset["content_inventory"],\n'
    "            )\n"
    "            for name, asset in sorted(ASSETS.items())\n"
    "        ],\n"
    "        constraints=InputSelectionConstraints(\n"
    "            maximum_modalities=MAX_MODALITIES,\n"
    "            maximum_total_cost=BUDGET,\n"
    "        ),\n"
    "    )\n"
    "    return run_iterative_selection(\n"
    "        request,\n"
    "        ITERATIVE_POLICY,\n"
    "        notebook_generator(selector_generate_reply, GENERATION_LOG),\n"
    "        load_notebook_evidence,\n"
    "        model_id=MODEL_NAME,\n"
    '        execution_kind="model",\n'
    "    )\n"
    "\n"
    "\n"
    "# Routing checks only: no model call. The final answer still uses "
    "answer_question\n"
    "# for ALL five conditions, preserving the exact input ordering and "
    "answer prompt.\n"
    "for modality, images in [\n"
    '    ("CRATER_CATALOG", 0),\n'
    '    ("OPTICAL_IMAGE", 2),\n'
    '    ("TOPOGRAPHY", 3),\n'
    '    ("MULTISPECTRAL_IMAGE", 0),\n'
    "]:\n"
    "    observation = load_notebook_evidence(InputDataModality(modality))\n"
    '    assert sum(b.type == "image" for b in observation.blocks) == '
    "images\n"
    '    assert all(ANSWER_POLICY not in (b.text or "") for b in '
    "observation.blocks)\n"
    'print("Two-step interface ready; real input routing verified without '
    'model inference.")',
    "experiment": "import gc\n"
    "import inspect\n"
    "import traceback\n"
    "from typing import Literal\n"
    "\n"
    "from pydantic import BaseModel, ConfigDict, Field, model_validator\n"
    "\n"
    "from autonomous_modality.protocol import ExperimentProtocol\n"
    "\n"
    'CONDITIONS = ["NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT", '
    '"AGENT_ITERATIVE"]\n'
    "\n"
    "\n"
    "class Attempt(BaseModel):\n"
    '    model_config = ConfigDict(extra="forbid", strict=True, '
    "allow_inf_nan=False)\n"
    '    record_version: Literal["colab-ap-four-modality-en-1"] = '
    '"colab-ap-four-modality-en-1"\n'
    "    run_sha256: str\n"
    "    model: str\n"
    "    case_id: str\n"
    "    question_id: str\n"
    "    question: str\n"
    '    condition: Literal["NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT", '
    '"AGENT_ITERATIVE"]\n'
    "    draw_index: int = Field(ge=0, le=4)\n"
    "    selection_seed: int | None = None\n"
    '    status: Literal["success", "truncated", "incomplete", "error"]\n'
    "    selected: list[str]\n"
    "    selection_rationale: str\n"
    "    iterative_selection: IterativeSelection | None = None\n"
    "    input_cost: int = Field(ge=0)\n"
    "    elapsed_seconds: float = Field(ge=0)\n"
    "    selection_seconds: float = Field(ge=0)\n"
    "    answer_seconds: float = Field(ge=0)\n"
    "    peak_gpu_gib: float | None = None\n"
    "    selection_generations: list[GenerationTrace]\n"
    "    answer_generations: list[GenerationTrace]\n"
    "    answer: str | None = None\n"
    '    finish_reason: Literal["eos", "length", "unknown"] | None = None\n'
    '    error_stage: Literal["selection", "answer"] | None = None\n'
    "    error_type: str | None = None\n"
    "    error: str | None = None\n"
    "    traceback: str | None = None\n"
    "\n"
    '    @model_validator(mode="after")\n'
    '    def check_record(self) -> "Attempt":\n'
    '        if self.condition != "RANDOM" and self.draw_index != 0:\n'
    '            raise ValueError("Non-random conditions cannot contain '
    'extra draws.")\n'
    '        if self.condition == "NO_DATA" and self.selected:\n'
    '            raise ValueError("NO_DATA must contain no additional '
    'evidence.")\n'
    "        if set(self.selected) - set(ASSETS) or len(set(self.selected)) "
    "!= len(self.selected):\n"
    '            raise ValueError("Record contains invalid modalities.")\n'
    '        if self.input_cost != sum(ASSETS[x]["cost"] for x in '
    "self.selected):\n"
    '            raise ValueError("Cost mismatch.")\n'
    '        if self.selected and self.condition in {"RANDOM", "AGENT", '
    '"AGENT_ITERATIVE"}:\n'
    "            valid_selection(self.selected)\n"
    '        if self.status != "error":\n'
    '            if self.condition == "ALL_AVAILABLE" and self.selected != '
    "sorted(ASSETS):\n"
    '                raise ValueError("ALL_AVAILABLE is missing '
    'modalities.")\n'
    '            if self.condition not in {"NO_DATA", "AGENT_ITERATIVE"} '
    "and not self.selected:\n"
    '                raise ValueError("A data condition is missing its '
    'selection.")\n'
    "            if not self.answer or not self.answer_generations:\n"
    '                raise ValueError("Answer text or generation record is '
    'missing.")\n'
    '            if self.status == "success" and self.finish_reason != '
    '"eos":\n'
    '                raise ValueError("An answer without EOS cannot be '
    'marked successful.")\n'
    '        if self.condition == "AGENT_ITERATIVE":\n'
    '            if self.status != "error" and self.iterative_selection is '
    "None:\n"
    '                raise ValueError("Missing iterative acquisition '
    'trace.")\n'
    "            if self.iterative_selection is not None:\n"
    "                trace = self.iterative_selection\n"
    "                if sorted(m.value for m in trace.accessed_modalities) "
    "!= self.selected:\n"
    '                    raise ValueError("Iterative access history differs '
    'from selected evidence.")\n'
    "                if trace.cumulative_cost != self.input_cost:\n"
    '                    raise ValueError("Iterative cumulative cost '
    'mismatch.")\n'
    '                if self.status != "error" and trace.status != '
    '"ready":\n'
    '                    raise ValueError("Cannot answer after failed '
    'acquisition.")\n'
    "        elif self.iterative_selection is not None:\n"
    '            raise ValueError("Iterative trace on another condition.")\n'
    "        return self\n"
    "\n"
    "\n"
    'protocol_path = PROJECT / "configs/experiment_protocol.json"\n'
    "protocol = "
    'ExperimentProtocol.model_validate_json(protocol_path.read_text(encoding="utf-8"))\n'
    "live_settings = {\n"
    '    "model": str(model.config._name_or_path),\n'
    '    "seed": SEED,\n'
    '    "budget": BUDGET,\n'
    '    "maximum_modalities": MAX_MODALITIES,\n'
    '    "random_draws": RANDOM_DRAWS,\n'
    '    "image_edge": IMAGE_EDGE,\n'
    '    "answer_tokens": ANSWER_TOKENS,\n'
    '    "answer_repetition_penalty": ANSWER_REPETITION_PENALTY,\n'
    '    "selector_tokens": SELECTOR_TOKENS,\n'
    '    "max_input_tokens": MAX_INPUT_TOKENS,\n'
    '    "iterative_selector_tokens": ITERATIVE_SELECTOR_TOKENS,\n'
    "}\n"
    "if any(CONFIG[key] != value for key, value in live_settings.items()):\n"
    '    raise ValueError("Settings changed; rerun from section 3 in '
    'order.")\n'
    "if MODEL_NAME not in protocol.models:\n"
    '    raise ValueError("Model differs from the project protocol.")\n'
    "torch.manual_seed(SEED)\n"
    "if torch.cuda.is_available():\n"
    "    torch.cuda.manual_seed_all(SEED)\n"
    "\n"
    "# Require inspect.getsource for code fingerprints; fail explicitly if "
    "source is unavailable.\n"
    "tracked_objects = (\n"
    "    sha256,\n"
    "    json_digest,\n"
    "    write_new_json,\n"
    "    portable_name,\n"
    "    generate_reply,\n"
    "    parse_agent_choice,\n"
    "    valid_selection,\n"
    "    select_inputs,\n"
    "    build_answer_messages,\n"
    "    answer_question,\n"
    "    Attempt.check_record,\n"
    "    build_evidence_content,\n"
    "    load_notebook_evidence,\n"
    "    select_inputs_iterative,\n"
    "    selector_generate_reply,\n"
    ")\n"
    "try:\n"
    "    function_sources = {x.__name__: inspect.getsource(x) for x in "
    "tracked_objects}\n"
    "except (OSError, TypeError) as exc:\n"
    '    raise RuntimeError("Cell source unavailable; rerun the notebook '
    'cells in order.") from exc\n'
    "\n"
    "metadata = {\n"
    '    "config": CONFIG,\n'
    '    "protocol": protocol.model_dump(mode="json"),\n'
    '    "protocol_sha256": sha256(protocol_path),\n'
    '    "functions": function_sources,\n'
    '    "iterative_policy": ITERATIVE_POLICY.model_dump(mode="json"),\n'
    '    "condition_roles": {"AGENT": "one_shot_ablation", '
    '"AGENT_ITERATIVE": "primary_agent"},\n'
    '    "schemas": {\n'
    "        cls.__name__: cls.model_json_schema()\n"
    "        for cls in (GenerationTrace, AgentChoice, Attempt, "
    "ViewReceipt, IterativeSelection)\n"
    "    },\n"
    '    "options": OPTIONS,\n'
    '    "model_config": json.loads(model.config.to_json_string()),\n'
    '    "generation_defaults": '
    "json.loads(model.generation_config.to_json_string()),\n"
    '    "processor_class": type(processor).__name__,\n'
    '    "image_processor": '
    "json.loads(processor.image_processor.to_json_string()),\n"
    '    "chat_template": getattr(processor, "chat_template", None),\n'
    '    "tokenizer_chat_template": getattr(processor.tokenizer, '
    '"chat_template", None),\n'
    '    "tokenizer_class": type(processor.tokenizer).__name__,\n'
    '    "environment": {\n'
    "        name: importlib.metadata.version(name)\n"
    "        for name in (\n"
    '            "torch",\n'
    '            "transformers",\n'
    '            "bitsandbytes",\n'
    '            "accelerate",\n'
    '            "numpy",\n'
    '            "Pillow",\n'
    '            "rasterio",\n'
    '            "matplotlib",\n'
    '            "pydantic",\n'
    "        )\n"
    "    },\n"
    '    "gpu": torch.cuda.get_device_name(0),\n'
    '    "cuda": torch.version.cuda,\n'
    '    "deterministic_algorithms": '
    "torch.are_deterministic_algorithms_enabled(),\n"
    "}\n"
    "# Recheck actual inputs; changed inputs require a new RUN_LABEL.\n"
    "if {name: sha256(path) for name, path in SOURCES.items()} != "
    "SOURCE_HASHES:\n"
    '    raise ValueError("Source files changed.")\n'
    "for name, digest in receipt.output_sha256.items():\n"
    "    if sha256(VIEW_DIR / name) != digest:\n"
    '        raise ValueError(f"Input view changed: {name}")\n'
    "# Within a RUN_LABEL, both models share questions, input bytes, "
    "prompts and random choices.\n"
    "# Across sessions, use persistent RESULTS_ROOT or restore the exported "
    "experiment directory.\n"
    "paired_design = {\n"
    '    "case_id": CASE_ID,\n'
    '    "source_sha256": SOURCE_HASHES,\n'
    '    "view_sha256": receipt.output_sha256,\n'
    '    "question_sha256": CONFIG["question_sha256"],\n'
    '    "budget": BUDGET,\n'
    '    "maximum_modalities": MAX_MODALITIES,\n'
    '    "answer_tokens": ANSWER_TOKENS,\n'
    '    "answer_repetition_penalty": ANSWER_REPETITION_PENALTY,\n'
    '    "selector_tokens": SELECTOR_TOKENS,\n'
    '    "answer_policy": ANSWER_POLICY,\n'
    '    "selector_task": SELECTOR_TASK,\n'
    '    "options": OPTIONS,\n'
    '    "iterative_policy": ITERATIVE_POLICY.model_dump(mode="json"),\n'
    '    "conditions": CONDITIONS,\n'
    '    "random_choices": {\n'
    '        q["question_id"]: [\n'
    '            select_inputs("RANDOM", q["text"], q["question_id"], i)[0] '
    "for i in range(RANDOM_DRAWS)\n"
    "        ]\n"
    "        for q in QUESTIONS\n"
    "    },\n"
    "}\n"
    "paired_path = RESULTS_ROOT / RUN_LABEL / "
    'f"paired-{CASE_ID}-b{BUDGET}.json"\n'
    "if paired_path.exists():\n"
    '    if json.loads(paired_path.read_text(encoding="utf-8")) != '
    "paired_design:\n"
    '        raise ValueError("Shared design or inputs differ between '
    'models; comparison is blocked.")\n'
    "else:\n"
    "    write_new_json(paired_path, paired_design)\n"
    'metadata["paired_design_sha256"] = sha256(paired_path)\n'
    'metadata_path = OUTPUT / "run_metadata.json"\n'
    "if metadata_path.exists():\n"
    '    if json.loads(metadata_path.read_text(encoding="utf-8")) != '
    "metadata:\n"
    '        raise ValueError("Code, model or configuration changed; use a '
    'new RUN_LABEL.")\n'
    "else:\n"
    '    if list(OUTPUT.glob("Q*__*.json")):\n'
    '        raise ValueError("Existing results have no matching '
    'configuration; use a new RUN_LABEL.")\n'
    "    write_new_json(metadata_path, metadata)\n"
    "RUN_SHA256 = sha256(metadata_path)\n"
    "\n"
    "# Test the largest input briefly; a full 1024-token answer may still "
    "exceed GPU memory.\n"
    "# Stop on failure; keep evidence and selection policy fixed across "
    "conditions.\n"
    'preflight_path = OUTPUT / "preflight.json"\n'
    "if not preflight_path.exists():\n"
    "    GENERATION_LOG.clear()\n"
    "    try:\n"
    "        generate_reply(\n"
    "            build_answer_messages(\n"
    '                "Suggest research using these inputs. Answer in '
    'English without measurements.",\n'
    "                [],\n"
    "                sorted(ASSETS),\n"
    "            ),\n"
    "            max_new_tokens=32,\n"
    "        )\n"
    "        preflight = {\n"
    '            "run_sha256": RUN_SHA256,\n'
    '            "status": "completed_input_smoke_test",\n'
    '            "generation": GENERATION_LOG[-1],\n'
    "        }\n"
    "    except Exception:\n"
    "        write_new_json(\n"
    "            preflight_path,\n"
    "            {\n"
    '                "run_sha256": RUN_SHA256,\n'
    '                "status": "error",\n'
    '                "generation": GENERATION_LOG[-1] if GENERATION_LOG '
    "else None,\n"
    "            },\n"
    "        )\n"
    "        raise\n"
    "    write_new_json(preflight_path, preflight)\n"
    "else:\n"
    "    preflight = "
    'json.loads(preflight_path.read_text(encoding="utf-8"))\n'
    '    if preflight.get("run_sha256") != RUN_SHA256 or '
    'preflight.get("status") == "error":\n'
    '        raise ValueError("Preflight changed or failed; resolve the '
    'issue and use a new RUN_LABEL.")\n'
    "\n"
    "for spec in QUESTIONS:\n"
    '    question_id = spec["question_id"]\n'
    '    question = spec["text"].replace("{crater}", CASE_NAME)\n'
    "    for condition in CONDITIONS:\n"
    '        for draw_index in range(RANDOM_DRAWS if condition == "RANDOM" '
    "else 1):\n"
    "            destination = OUTPUT / "
    'f"{question_id}__{condition}__r{draw_index + 1:02d}.json"\n'
    "            selection_seed = (\n"
    '                scenario_seed(SEED, f"{CASE_ID}-{question_id}", '
    "BUDGET, draw_index)\n"
    '                if condition == "RANDOM"\n'
    "                else None\n"
    "            )\n"
    "            if destination.exists():\n"
    "                old = "
    'Attempt.model_validate_json(destination.read_text(encoding="utf-8"))\n'
    "                actual = (\n"
    "                    old.run_sha256,\n"
    "                    old.model,\n"
    "                    old.case_id,\n"
    "                    old.question_id,\n"
    "                    old.condition,\n"
    "                    old.draw_index,\n"
    "                    old.selection_seed,\n"
    "                )\n"
    "                expected = (\n"
    "                    RUN_SHA256,\n"
    "                    MODEL_NAME,\n"
    "                    CASE_ID,\n"
    "                    question_id,\n"
    "                    condition,\n"
    "                    draw_index,\n"
    "                    selection_seed,\n"
    "                )\n"
    "                if actual != expected:\n"
    '                    raise ValueError(f"Existing result identity '
    'mismatch: {destination.name}")\n'
    "                print(\n"
    '                    "Keeping existing attempt (including failures):", '
    "destination.name, old.status\n"
    "                )\n"
    "                continue\n"
    "\n"
    "            GENERATION_LOG.clear()\n"
    "            started = stage_started = time.perf_counter()\n"
    '            stage = "selection"\n'
    '            selected, rationale = [], ""\n'
    "            iterative_selection = None\n"
    "            selection_seconds = answer_seconds = 0.0\n"
    "            selection_generations, answer_generations = [], []\n"
    "            outcome = {}\n"
    "            try:\n"
    '                if condition == "AGENT_ITERATIVE":\n'
    "                    iterative_selection = "
    "select_inputs_iterative(question, question_id)\n"
    "                    selected = sorted(m.value for m in "
    "iterative_selection.accessed_modalities)\n"
    '                    rationale = " | ".join(\n'
    "                        t.action.reason for t in "
    "iterative_selection.turns if t.action is not None\n"
    "                    )\n"
    '                    if iterative_selection.status != "ready":\n'
    "                        raise ValueError(\n"
    '                            f"{iterative_selection.stop_reason}: '
    '{iterative_selection.error}"\n'
    "                        )\n"
    "                else:\n"
    "                    selected, rationale = select_inputs(\n"
    "                        condition, question, question_id, "
    "draw_index=draw_index\n"
    "                    )\n"
    "                selection_seconds = time.perf_counter() - "
    "stage_started\n"
    "                selection_generations = copy.deepcopy(GENERATION_LOG)\n"
    "                GENERATION_LOG.clear()\n"
    '                stage, stage_started = "answer", time.perf_counter()\n'
    "                if iterative_selection is not None:\n"
    "                    for turn in iterative_selection.turns:\n"
    "                        if turn.observation is not None:\n"
    "                            current = "
    "load_notebook_evidence(turn.observation.modality)\n"
    "                            if current != turn.observation:\n"
    '                                raise ValueError("Acquired evidence '
    'changed before answering.")\n'
    "                answer = answer_question(question, "
    'spec["answer_requirements"], selected)\n'
    "                answer_seconds = time.perf_counter() - stage_started\n"
    "                answer_generations = copy.deepcopy(GENERATION_LOG)\n"
    "                if not answer_generations or not answer.strip():\n"
    '                    raise ValueError("EMPTY_OR_UNLOGGED_ANSWER")\n'
    '                finish = answer_generations[-1]["finish_reason"]\n'
    "                outcome = {\n"
    '                    "status": {"eos": "success", "length": '
    '"truncated"}.get(finish, "incomplete"),\n'
    '                    "answer": answer,\n'
    '                    "finish_reason": finish,\n'
    "                }\n"
    "            except Exception as exc:\n"
    '                if stage == "selection":\n'
    "                    selection_seconds = time.perf_counter() - "
    "stage_started\n"
    "                    selection_generations = "
    "copy.deepcopy(GENERATION_LOG)\n"
    "                else:\n"
    "                    answer_seconds = time.perf_counter() - "
    "stage_started\n"
    "                    answer_generations = "
    "copy.deepcopy(GENERATION_LOG)\n"
    "                outcome = {\n"
    '                    "status": "error",\n'
    '                    "error_stage": stage,\n'
    '                    "error_type": type(exc).__name__,\n'
    '                    "error": str(exc),\n'
    '                    "traceback": traceback.format_exc(),\n'
    "                }\n"
    "            peaks = [\n"
    '                g["peak_gpu_gib"]\n'
    "                for g in selection_generations + answer_generations\n"
    '                if g.get("peak_gpu_gib") is not None\n'
    "            ]\n"
    "            result = Attempt(\n"
    "                run_sha256=RUN_SHA256,\n"
    "                model=MODEL_NAME,\n"
    "                case_id=CASE_ID,\n"
    "                question_id=question_id,\n"
    "                question=question,\n"
    "                condition=condition,\n"
    "                draw_index=draw_index,\n"
    "                selection_seed=selection_seed,\n"
    "                selected=selected,\n"
    "                selection_rationale=rationale,\n"
    "                iterative_selection=iterative_selection,\n"
    '                input_cost=sum(ASSETS[x]["cost"] for x in selected),\n'
    "                elapsed_seconds=time.perf_counter() - started,\n"
    "                selection_seconds=selection_seconds,\n"
    "                answer_seconds=answer_seconds,\n"
    "                peak_gpu_gib=max(peaks, default=None),\n"
    "                selection_generations=selection_generations,\n"
    "                answer_generations=answer_generations,\n"
    "                **outcome,\n"
    "            )\n"
    '            temporary = destination.with_suffix(".json.tmp")\n'
    '            write_new_json(temporary, result.model_dump(mode="json"))\n'
    "            if destination.exists():\n"
    '                raise RuntimeError("Do not run concurrent notebooks '
    'writing to the same RUN_LABEL.")\n'
    "            temporary.replace(destination)\n"
    "            print(\n"
    "                question_id,\n"
    "                condition,\n"
    "                draw_index + 1,\n"
    "                result.status,\n"
    "                selected,\n"
    "                result.finish_reason or result.error,\n"
    "            )\n"
    "            GENERATION_LOG.clear()\n"
    "            gc.collect()\n"
    "            torch.cuda.empty_cache()",
}

## Run all three targets
Starts GPU inference, including an all-modality preflight per target. Failed preflight stops the batch.

In [ ]:
import gc
import hashlib
import json
from collections import Counter

from IPython import get_ipython

EXECUTION_SOURCES_SHA256 = hashlib.sha256(
    json.dumps(EXECUTION_SOURCES, sort_keys=True).encode("utf-8")
).hexdigest()
run_root = RESULTS_ROOT / RUN_LABEL
run_root.mkdir(parents=True, exist_ok=True)
source_path = run_root / "execution_sources.json"
if source_path.exists():
    if json.loads(source_path.read_text()) != EXECUTION_SOURCES:
        raise ValueError("Execution code changed; use a new RUN_LABEL.")
else:
    with source_path.open("x", encoding="utf-8") as stream:
        json.dump(EXECUTION_SOURCES, stream, indent=2)

shell = get_ipython()
if shell is None:
    raise RuntimeError("Run this notebook in Colab/IPython.")
for TARGET_ID in TARGET_IDS:
    print("Starting development target:", TARGET_ID)
    for stage_name in ("target_setup", "selector", "answer", "iterative", "experiment"):
        # IPython preserves inspectable sources for existing provenance checks.
        execution = shell.run_cell(EXECUTION_SOURCES[stage_name])
        execution.raise_error()
    # The stages above publish these validated objects in the notebook namespace.
    Attempt = globals()["Attempt"]
    OUTPUT = globals()["OUTPUT"]
    CASE_NAME = globals()["CASE_NAME"]
    MODEL_TAG = globals()["MODEL_TAG"]
    target_records = [
        Attempt.model_validate_json(p.read_text(encoding="utf-8"))
        for p in sorted(OUTPUT.glob("Q*__*__r*.json"))
    ]
    if len(target_records) != 36:
        raise RuntimeError(f"Incomplete target {TARGET_ID}: {len(target_records)}/36")
    print(CASE_NAME, "36/36", dict(Counter(r.status for r in target_records)))
    GENERATION_LOG.clear()
    gc.collect()
    torch.cuda.empty_cache()
print("Batch finished. Failed/truncated attempts remain recorded, without retries.")

## Summary and export

In [ ]:
from collections import Counter

all_records = []
for target in TARGET_IDS:
    folder = RESULTS_ROOT / RUN_LABEL / MODEL_TAG / f"crater-{target}" / f"budget-{BUDGET}"
    values = [
        Attempt.model_validate_json(p.read_text(encoding="utf-8"))
        for p in sorted(folder.glob("Q*__*__r*.json"))
    ]
    print(target, len(values), "/36", dict(Counter(r.status for r in values)))
    all_records.extend(values)
print("Total:", len(all_records), "/108")
print("Statuses:", dict(Counter(r.status for r in all_records)))
print("Conditions:", dict(Counter(r.condition for r in all_records)))
for condition in ("AGENT", "AGENT_ITERATIVE"):
    print(
        condition,
        dict(
            Counter(
                tuple(r.selected)
                for r in all_records
                if r.condition == condition and r.status == "success"
            )
        ),
    )
print("Selection frequency is descriptive; evaluate A/P and quality separately.")

In [ ]:
import shutil
from datetime import UTC, datetime

from google.colab import files

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
archive = shutil.make_archive(
    str(RESULTS_ROOT / f"{RUN_LABEL}-{stamp}_results"),
    "zip",
    root_dir=RESULTS_ROOT,
    base_dir=RUN_LABEL,
)
files.download(archive)